In [1]:
from primaite.agents.git.git_agent import GITAgent
from primaite.agents.git.git_policy import GITPolicy

from primaite.agents.llm.utils import get_obs_act_history_str, obs_diff
from primaite.environment import EnvironmentState

/home/jonathan/projects/primaite/PrimAITE/src/primaite/agents/git/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jonathan/projects/primaite/PrimAITE/src/primaite/agents/git/aegis/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
2024-08-07 13:27:13.347194: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-07 13:27:14.670679: W tensorflow/compiler/tf2t

In [2]:
agent = GITAgent(training_config_path='../../config/_package_data/training/git.yaml', lay_down_config_path='../../config/_package_data/lay_down/lay_down_config_6_data_manipulation.yaml')

2024-08-07 13:27:16,950: Using: AgentFramework.CUSTOM, AgentIdentifier.GIT, ActionType.NODE, observation_space=NODE_LINK_TABLE, 512 episodes @ 150 steps
2024-08-07 13:27:17,134: Environment configuration loaded
/home/jonathan/projects/primaite/PrimAITE/src/primaite/agents/git/aegis/lib/python3.10/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


[{'role': 'user', 'content': 'hack'}]


RuntimeError: CUDA error: invalid device ordinal
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


<Figure size 640x480 with 0 Axes>

In [3]:
import torch
torch.cuda.is_available()

True

In [ ]:
from primaite.agents.git.git_policy import LLM_PROMPT

In [ ]:
node_str = "\n".join(f"{key}: {value}" for key, value in agent._env.nodes.items()) # ID: NODE_NAME

nodes_index = agent._env.services_list
services_str = '\n'.join(f'{i}: {service}' for i, service in enumerate(nodes_index)) # ID: SERVICE

nodes_table = agent.env_history[0].nodes_table.rename(columns={'TCP Service State': 'TCP', 'TCP_SQL Service State': 'TCP_SQL', 'UDP Service State': 'UDP'})
node_services = {}
for idx in range(len(nodes_table)):
    node = nodes_table.iloc[idx, :]
    node_services[node['Name']] = []
    
    if node['TCP'] != '-':
        node_services[node['Name']].append(nodes_index.index('TCP'))
    if node['TCP_SQL'] != '-':
        node_services[node['Name']].append(nodes_index.index('TCP_SQL'))
    if node['UDP'] != '-':
        node_services[node['Name']].append(nodes_index.index('UDP'))
        
node_services_str = "\n".join(f"{key}: {', '.join(str(val + 1) for val in value) if value else 'NONE'}" for key, value in node_services.items()) # NODE_NAME: SERVICES

obs_act_history_str = get_obs_act_history_str(agent.env_history, env=agent._env)
if not obs_act_history_str:
    obs_act_history_str = 'No observation history yet...'
obs_diff_str = obs_diff(EnvironmentState(env=agent._env))

NameError: name 'agent' is not defined

In [ ]:
prompt=LLM_PROMPT.format(node_ids=node_str, services=services_str, node_services=node_services_str, obs_act_history=obs_act_history_str, current_obs_diff=obs_diff_str)

NameError: name 'node_str' is not defined

In [ ]:
print(prompt)

NameError: name 'prompt' is not defined

In [ ]:
embeddings = agent._agent.llm.get_embeddings(prompt=prompt)

[{'role': 'user', 'content': "Your job is to defend the network against attacks. Given the provided network graph tokens, please choose one action to execute within the environment. Baring in mind that you will be rewarded for taking the most suitible action in a timely manner and with consideration for what nodes might take the highest priority.\n\nThe nodes in the network are:\n1: CLIENT_1\n2: CLIENT_2\n3: SWITCH_1\n4: SECURITY_SUITE\n5: MANAGEMENT_CONSOLE\n6: SWITCH_2\n7: WEB_SERVER\n8: DATABASE_SERVER\n9: BACKUP_SERVER\n\nThe available services and their respective nodes are:\nTCP, TCP_SQL, UDP\n\nThe actions you could take are laid out below:\nTURN_ON - Turn on a node\nTURN_OFF - Turn off a node\nRESET - Reset a node\nPATCH_HARDWARE - Patch a nodes hardware\nPATCH_SERVICE - Patch a nodes service\n\nYou must always state the action name in full and which node number this action is to be applied to. If the action is a service patch, always specify which service to patch.\n\nHere are

In [10]:
token_ids, probs = agent._agent.llm.generate_from_embeddings(text_embeddings=embeddings, grad=False, max_new_tokens=10)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


In [11]:
print(agent._agent.llm.tokenizer.decode(token_ids))

<|im_start|>assistant
To defend the network against attacks


In [3]:
agent.learn()

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
2024-08-07 08:24:46.658891: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-07 08:24:47.397799: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 1 has a total capacity of 10.75 GiB of which 17.62 MiB is free. Including non-PyTorch memory, this process has 10.73 GiB memory in use. Of the allocated memory 10.17 GiB is allocated by PyTorch, and 382.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)